# Notebook 2: Wind Power Reliability Analysis

**Goal**: Understand how reliably wind power can meet electricity demand based on historical January 2024 actual generation data, and recommend how many MW of wind power can be reliably expected.

## Analytical Approach

**Assumption**: "Reliable" means *always available at or above this level*, i.e., we want to find a level X such that wind generation seldom falls below it.

**First principles reasoning**:
- Wind generation is highly variable — it can drop to near-zero during wind droughts or spike well above mean during storms.
- For grid planning purposes, the *firm capacity* of wind is typically anchored to a low percentile (e.g. P5 or P10 of generation) — the level that is exceeded 90–95% of the time.
- The P10 value means: "In 90% of 30-minute periods, actual wind generation was above this level."
- This is a conservative but operationally meaningful baseline for planning purposes.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5)})

BASE = 'https://data.elexon.co.uk/bmrs/api/v1'
print('Ready')

## 1. Fetch All January 2024 Actuals

In [ ]:
def fetch_actuals_jan2024():
    all_records = []
    for start, end in [('2024-01-01','2024-01-08'),('2024-01-08','2024-01-15'),
                       ('2024-01-15','2024-01-22'),('2024-01-22','2024-01-31')]:
        r = requests.get(
            f'{BASE}/datasets/FUELHH/stream',
            params={'settlementDateFrom': start, 'settlementDateTo': end,
                    'fuelType': 'WIND', 'format': 'json'}, timeout=60
        )
        r.raise_for_status()
        all_records.extend(r.json())
    df = pd.DataFrame(all_records)[['startTime', 'generation']]
    df['startTime'] = pd.to_datetime(df['startTime'], utc=True)
    df = df[(df['startTime'] >= '2024-01-01') & (df['startTime'] < '2024-02-01')]
    df = df.sort_values('startTime').drop_duplicates('startTime').reset_index(drop=True)
    df.columns = ['time', 'generation_mw']
    return df

df = fetch_actuals_jan2024()
print(f'Total half-hourly readings: {len(df)}')
print(f'Period:  {df["time"].min()} → {df["time"].max()}')
df.head()

## 2. Summary Statistics

In [ ]:
stats = df['generation_mw'].describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
print('=== Actual Wind Generation Statistics — January 2024 ===')
print(stats.round(0).to_string())
print(f'\nInstalled UK wind capacity (approx. Jan 2024): ~29,000 MW')
print(f"Capacity factor range: {df['generation_mw'].min()/29000*100:.1f}% – {df['generation_mw'].max()/29000*100:.1f}%")

## 3. Distribution Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('UK Wind Power Actual Generation — January 2024', fontsize=14, fontweight='bold')

# Histogram
axes[0].hist(df['generation_mw'], bins=50, color='#38bdf8', alpha=0.8, edgecolor='white', linewidth=0.3)
for pct, label, color in [(10, 'P10', '#f87171'), (50, 'P50', '#fbbf24'), (90, 'P90', '#34d399')]:
    v = np.percentile(df['generation_mw'], pct)
    axes[0].axvline(v, color=color, linestyle='--', linewidth=1.5, label=f'P{pct}: {v:.0f} MW')
axes[0].set_xlabel('Wind Generation (MW)')
axes[0].set_ylabel('Frequency (half-hour periods)')
axes[0].set_title('Generation Distribution')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Time series
axes[1].fill_between(df['time'], df['generation_mw'], alpha=0.35, color='#38bdf8')
axes[1].plot(df['time'], df['generation_mw'], color='#38bdf8', linewidth=0.8)
p10 = np.percentile(df['generation_mw'], 10)
axes[1].axhline(p10, color='#f87171', linestyle='--', linewidth=1.5, label=f'P10: {p10:.0f} MW')
axes[1].set_xlabel('Date (January 2024)')
axes[1].set_ylabel('Wind Generation (MW)')
axes[1].set_title('Actual Generation Time Series')
axes[1].legend()
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.savefig('../notebooks/generation_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Percentile Table — What Level Is Reliably Available?

In [ ]:
percentiles = [1, 5, 10, 20, 25, 30, 50, 70, 75, 80, 90, 95, 99]
pct_values  = np.percentile(df['generation_mw'], percentiles)

pct_df = pd.DataFrame({
    'Percentile': [f'P{p}' for p in percentiles],
    'Generation (MW)': pct_values.round(0).astype(int),
    'Exceeded (% of time)': [f'{100-p}%' for p in percentiles],
})
print('=== Percentile Summary ===')
print(pct_df.to_string(index=False))
print()
p10 = int(np.percentile(df['generation_mw'], 10))
p5  = int(np.percentile(df['generation_mw'], 5))
print(f'P5  = {p5:,} MW  → exceeded 95% of half-hour periods')
print(f'P10 = {p10:,} MW  → exceeded 90% of half-hour periods')

## 5. Low-Generation Events (Near-Zero Wind)

In [ ]:
threshold = 3000  # MW
low_events = df[df['generation_mw'] < threshold]
print(f'=== Periods with generation < {threshold:,} MW ===')
print(f'Count: {len(low_events)} half-hour periods ({len(low_events)/len(df)*100:.1f}% of total)')
if len(low_events):
    print(f'Minimum observed: {df["generation_mw"].min():,} MW')
    print()
    print(low_events[['time', 'generation_mw']].to_string(index=False))
else:
    print('No such events in Jan 2024 — wind never dropped below threshold!')

# Plot low-generation periods highlighted
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['time'], df['generation_mw'], color='#38bdf8', linewidth=0.8, label='Actual generation')
ax.fill_between(df['time'], df['generation_mw'], where=(df['generation_mw'] < threshold),
                color='#f87171', alpha=0.5, label=f'< {threshold//1000}k MW (low wind)')
ax.axhline(threshold, color='#f87171', linestyle='--', linewidth=1.2)
ax.set_title('January 2024 Wind Generation — Low-Wind Events Highlighted')
ax.set_xlabel('Date')
ax.set_ylabel('MW')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
plt.tight_layout()
plt.savefig('../notebooks/low_generation_events.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Variability by Hour and Day of Week

In [ ]:
df['hour']       = df['time'].dt.hour
df['day_of_week'] = df['time'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Wind Generation Variability Breakdown', fontsize=14, fontweight='bold')

# Box plot by hour
hour_groups = [df[df['hour']==h]['generation_mw'].values for h in range(24)]
axes[0].boxplot(hour_groups, positions=range(24), widths=0.7,
                patch_artist=True,
                boxprops=dict(facecolor='#38bdf850', color='#38bdf8'),
                medianprops=dict(color='#fbbf24', linewidth=2),
                whiskerprops=dict(color='#38bdf8'),
                flierprops=dict(marker='o', markerfacecolor='#f8718150', markersize=2))
axes[0].set_xlabel('Hour of Day (UTC)')
axes[0].set_ylabel('Wind Generation (MW)')
axes[0].set_title('Generation by Hour of Day')
axes[0].set_xticks(range(0, 24, 3))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Mean generation by day
daily = df.groupby(df['time'].dt.date)['generation_mw'].agg(['mean','min','max'])
axes[1].fill_between(range(len(daily)), daily['min'], daily['max'], alpha=0.25, color='#38bdf8', label='Min–Max range')
axes[1].plot(range(len(daily)), daily['mean'], color='#38bdf8', linewidth=2, label='Daily mean')
axes[1].set_xticks(range(0, len(daily), 3))
axes[1].set_xticklabels([str(d) for d in daily.index[::3]], rotation=30, fontsize=8)
axes[1].set_title('Daily Mean / Min / Max Generation')
axes[1].set_ylabel('Wind Generation (MW)')
axes[1].legend()
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.savefig('../notebooks/variability_breakdown.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Recommendation

### How many MW of wind can we reliably expect?

**Framework**: We define "reliable" as: *the level of wind generation that is exceeded in at least 90% of all observed half-hour settlement periods (P10)*.

This is the standard industry approach for calculating **firm capacity** or **de-rated capacity** of intermittent generators.

**Reasoning**:
- Using the **mean** (~11,000–13,000 MW) would be misleading — half the time generation is below this.
- Using the **minimum** is too conservative — near-zero events are rare and grid operators have other backup sources.
- Using **P10** provides a balance: it excludes the worst 10% of scenarios (which should be covered by dispatchable backup), while representing what the system can *bank on* in normal operations.
- **P5** (~X MW) gives even higher confidence (95% of periods), suitable for long-term capacity adequacy studies.

**Caveats**:
- January is a winter month and likely above the annual average for UK wind (winter storms bring high generation).
- A multi-year, multi-month analysis would give a more robust estimate.
- The UK installed wind capacity (onshore + offshore ~29 GW as of Jan 2024) matters — capacity factors fluctuate with fleet expansion.

In [ ]:
p5  = int(np.percentile(df['generation_mw'], 5))
p10 = int(np.percentile(df['generation_mw'], 10))
p50 = int(np.percentile(df['generation_mw'], 50))
mean_gen = int(df['generation_mw'].mean())

print('╔══════════════════════════════════════════════════════════╗')
print('║       WIND POWER RELIABILITY RECOMMENDATION              ║')
print('╟──────────────────────────────────────────────────────────╢')
print(f'║  Conservative (P5  — 95% reliability): {p5:>8,} MW   ║')
print(f'║  Recommended  (P10 — 90% reliability): {p10:>8,} MW   ║')
print(f'║  Median       (P50 — 50% reliability): {p50:>8,} MW   ║')
print(f'║  Mean generation (Jan 2024 average):   {mean_gen:>8,} MW   ║')
print('╟──────────────────────────────────────────────────────────╢')
print('║  RECOMMENDATION: Plan on ~P10 MW of firm wind capacity.  ║')
print('║  This level is available in ≥90% of all 30-min periods,  ║')
print('║  providing a robust planning baseline for grid operators. ║')
print('╚══════════════════════════════════════════════════════════╝')